# Logistic Regression
## Real-world scenario: Will a customer default on a loan?

A bank wants to predict whether a loan applicant will **default (1)** or **repay (0)**. The answer is a yes/no label, so this is a **binary classification** problem - the classic use of Logistic Regression, which outputs a probability between 0 and 1.

### Step 1 - Import the libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix,
                             classification_report)

np.random.seed(42)

### Step 2 - Create a small, realistic dataset
50 applicants with age, annual income, existing debt and credit score. We again add a few missing values and a duplicate to clean later.

In [ ]:
n = 50
age          = np.random.randint(21, 65, n)
income       = np.random.randint(20000, 120000, n)
debt         = np.random.randint(0, 40000, n)
credit_score = np.random.randint(300, 850, n)

# Higher debt & lower credit score -> more likely to default
risk = (debt / 40000) - (credit_score - 300) / 550 + np.random.normal(0, 0.25, n)
default = (risk > risk.mean()).astype(int)   # 1 = default, 0 = repaid

df = pd.DataFrame({
    'age': age, 'income': income, 'debt': debt,
    'credit_score': credit_score, 'default': default
})

# Add messy data on purpose
df.loc[5, 'income'] = np.nan
df.loc[9, 'credit_score'] = np.nan
df = pd.concat([df, df.iloc[[1]]], ignore_index=True)  # duplicate
df.head()

### Step 3 - Explore the data

In [ ]:
print('Shape:', df.shape)
print('\nMissing values:\n', df.isnull().sum())
print('\nDuplicates:', df.duplicated().sum())
print('\nClass balance (0=repaid, 1=default):\n', df['default'].value_counts())

### Step 4 - Clean the data

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)         # remove duplicates
df['income'] = df['income'].fillna(df['income'].median())
df['credit_score'] = df['credit_score'].fillna(df['credit_score'].median())
print('Missing after cleaning:', df.isnull().sum().sum())

### Step 5 - Features (X) and target (y)

In [ ]:
X = df[['age', 'income', 'debt', 'credit_score']]
y = df['default']

### Step 6 - Train / test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)  # keep class ratio
print('Train:', len(X_train), '| Test:', len(X_test))

### Step 7 - Scale the features
Logistic Regression works better when features are on a similar scale (income is in the tens of thousands, age is under 100). We fit the scaler on the training data only, then apply it to both sets.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

### Step 8 - Train the model

In [ ]:
model = LogisticRegression()
model.fit(X_train_scaled, y_train)

### Step 9 - Evaluate
For classification we look at accuracy, the confusion matrix (correct vs wrong per class) and precision/recall/F1.

In [ ]:
y_pred = model.predict(X_test_scaled)

print('Accuracy:', round(accuracy_score(y_test, y_pred), 3))
print('\nConfusion matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification report:\n',
      classification_report(y_test, y_pred, zero_division=0))

### Step 10 - Predict for a new applicant
Age 40, income 45,000, debt 30,000, credit score 520:

In [ ]:
new_applicant = pd.DataFrame({'age': [40], 'income': [45000],
                              'debt': [30000], 'credit_score': [520]})
new_scaled = scaler.transform(new_applicant)
proba = model.predict_proba(new_scaled)[0][1]
print(f'Probability of default: {proba:.1%}')
print('Decision:', 'DEFAULT risk' if proba > 0.5 else 'Likely to repay')